In [45]:
import requests
import json
from tqdm import tqdm
import pandas as pd
from pathlib import Path

In [8]:
with open("./test_scrap_UE/survey_dataset_urls.json", "r", encoding="utf-8") as f:
    survey_dataset_urls = json.load(f)

In [18]:
distributions = []
idx_val = 0
for idx, url in tqdm(survey_dataset_urls.items()):
    suffix_ds = url.split("/")[-1].lower()
    try:
        resp = requests.get(
            f"https://data.europa.eu/api/hub/search/datasets/{suffix_ds}"
        )
        distributions.extend(resp.json()["result"]["distributions"])
    except Exception as e:
        print(f"Error scraping {idx}: {e}")

 40%|███▉      | 395/994 [00:54<01:19,  7.51it/s]

Error scraping 2019: Expecting value: line 1 column 1 (char 0)


 42%|████▏     | 414/994 [00:57<01:11,  8.13it/s]

Error scraping 2034: Expecting value: line 1 column 1 (char 0)


100%|██████████| 994/994 [02:25<00:00,  6.85it/s]


In [ ]:
with open("./test_scrap_UE/distributions_metadata.json", "w", encoding="utf-8") as f:
    json.dump(distributions, f, indent=2)

## download the excel files

In [3]:
with open("./test_scrap_UE/distributions_metadata.json", "r", encoding="utf-8") as f:
    distributions = json.load(f)

In [4]:
len(distributions)

6191

In [ ]:
distributions_df = pd.json_normalize(distributions, sep="_")

In [33]:
distributions_df["access_url"] = distributions_df["access_url"].apply(lambda x: x[0])

In [37]:
distributions_df[distributions_df["access_url"].str.contains("volume_B.")]

,id,media_type,issued,access_url,title_ro,title_ga,title_pt,title_lv,title_it,title_hu,...,format_id,format_label,format_resource,rights_resource,title_uk,description_uk,modified,status_label,status_resource,download_url


In [41]:
no_nandistributions_df = distributions_df.dropna(subset=["title_fr"])

In [ ]:
volume_b_docs_df = no_nandistributions_df[
    (
        (no_nandistributions_df["title_fr"].str.contains("volume_B."))
        | (no_nandistributions_df["title_fr"].str.contains("volume B."))
    )
    & ~(no_nandistributions_df["title_fr"].str.contains("volume_BP"))
]

In [71]:
# Create folder for downloads
downloads_folder = Path("./excels")
downloads_folder.mkdir(exist_ok=True)

# Download each file (Excel or zip)
for _, row in tqdm(
    volume_b_docs_df.iterrows(), total=len(volume_b_docs_df), desc="Downloading"
):
    url = row["access_url"]
    title = row["title_fr"].replace(" ", "_").replace("/", "_")

    # Get extension from title_fr (e.g., "volume_B.xls" -> "xls", "volume_B.zip" -> "zip")
    extension = row["title_fr"].split(".")[-1].lower()
    if extension not in ("xls", "xlsx", "zip"):
        extension = "xlsx"  # Default fallback

    filename = f"{title.rsplit('.', 1)[0]}.{extension}"
    filepath = downloads_folder / filename

    # Skip if already downloaded
    if filepath.exists():
        continue

    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        with open(filepath, "wb") as f:
            f.write(resp.content)
    except Exception as e:
        print(f"Error downloading {url}: {e}")


Downloading: 100%|██████████| 131/131 [04:46<00:00,  2.19s/it]
